# FRI-BCI Phase 2 · Cross-Session Stieger EEG Scaling Campaign

This notebook runs resumable Phase 2 training on Google Colab for the Stieger et al. (2021) corpus (`doi:10.6084/m9.figshare.13123148`).
- **Local disk staging:** 100 Hz 9-channel strictly causal `.npz` shards live in `/content/phase2_shards` (raw 600 MB `.mat` files are deleted immediately after verified conversion).
- **Drive persistence:** Checkpoints (`ckpt_latest.pt`, `ckpt_best.pt`), `job_state.json`, and `campaign_ledger.json` are copied to Drive and SHA-256 verified before recording completion.
- **Resumable execution:** After any Colab runtime reset or timeout, re-run Cells 2–3 and Cell 5 (`next-job --execute`) to continue from the last verified checkpoint.
- **Zero test leakage:** Only `source_train` (`44` subjects: `10%/30%/100%` nested prefixes) and `dev` (`8` subjects, including `S1` & `S2`) are staged here; `test_subjects` remain untouched until Work Package 4.

In [ ]:
# Cell 2: Clone/checkout repo, verify core Phase 2 file hashes, install dependencies, and inspect runtime
from pathlib import Path
import hashlib, os, platform, shutil, subprocess, sys

REPO = Path('/content/FRI-BCI-Phase2')
URL = 'https://github.com/Mohnclizzy/FRI-BCI-ML-RL-Decoder-Personal.git'
REF = 'codex/robot-learning-design'
PINNED_COMMIT = None  # Optionally pin a specific commit hash

if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', REF, URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', PINNED_COMMIT or f'origin/{REF}'], check=True)
if PINNED_COMMIT:
    subprocess.run(['git', '-C', str(REPO), 'checkout', PINNED_COMMIT], check=True)

os.chdir(REPO)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repo HEAD:', commit)

EXPECTED = {
    "eeg_decoder/neural.py": "6d495b1673317a3a1bdad3c2795ac5d1257e98fd3f7b440a58aa6ff07abb406a",
    "eeg_decoder/stieger.py": "874c7f90d119badf9a35e44421ca0b69112578e574381652d65705d6a334fce9",
    "eeg_decoder/phase2_train.py": "7a4da66192245cc6943b4824ea072f1e91ccc95942f3c7a1a6573fc889dd5a1e",
    "configs/phase2_stieger_split.json": "8b4012a361e847e574bc8912a36620d167daf292092b67a196055cfd7fb9eead",
    "configs/phase2_training.json": "dd36391fd6308b593a73fac06ac05fe688ca83e70fdb74a19b329460414e11be"
}
for rel, want in EXPECTED.items():
    got = hashlib.sha256(Path(rel).read_text(encoding='utf-8').encode()).hexdigest()
    assert got == want, f'{rel}: hash mismatch ({got} != {want}); update PINNED_COMMIT or EXPECTED'
print('All Phase 2 code/config SHA-256 hashes verified.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', 'pyriemann', 'requests'], check=True)
import torch
disk = shutil.disk_usage('/content')
print(f'Python {platform.python_version()} | Torch {torch.__version__} | CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory / (1024**3):.2f} GiB | bf16={torch.cuda.is_bf16_supported()}')
print(f'CPU cores: {os.cpu_count()} | /content free disk: {disk.free / (1024**3):.2f} GiB')

In [ ]:
# Cell 3: Mount Google Drive, verify manifests, and show current Phase 2 campaign ledger
from pathlib import Path
import os, subprocess, sys
from google.colab import drive

MOUNT = '/content/fri_drive'
if not os.path.ismount(MOUNT):
    drive.mount(MOUNT)
assert os.path.ismount(MOUNT), 'Drive mount failed'

DRIVE_WORK = Path(MOUNT) / 'MyDrive/FRI_BCI/phase2_stieger_scaling'
DRIVE_SHARDS = DRIVE_WORK / 'shards_100hz_9ch'
LOCAL_SHARDS = Path('/content/phase2_shards')
LOCAL_RAW = Path('/content/phase2_raw_tmp')
LOCAL_WORK = Path('/content/phase2_work')
for p in (DRIVE_WORK, DRIVE_SHARDS, LOCAL_SHARDS, LOCAL_RAW, LOCAL_WORK):
    p.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, '-m', 'eeg_decoder.phase2_train', 'status',
    '--shard-dir', str(LOCAL_SHARDS),
    '--work-dir', str(LOCAL_WORK),
    '--drive-dir', str(DRIVE_WORK),
], check=True)

In [ ]:
# Cell 4: Stage 3 audited dev shards and run GPU throughput + Drive interrupt/resume smoke benchmark
import json, subprocess, sys

subprocess.run([
    sys.executable, '-u', '-m', 'eeg_decoder.phase2_train', 'ingest',
    '--sessions', 'S1_Session_1', 'S1_Session_7', 'S2_Session_1',
    '--shard-dir', str(LOCAL_SHARDS),
    '--raw-dir', str(LOCAL_RAW),
    '--drive-shard-dir', str(DRIVE_SHARDS),
], check=True)

smoke_common = [
    sys.executable, '-u', '-m', 'eeg_decoder.phase2_train', 'train',
    '--shard-dir', str(LOCAL_SHARDS),
    '--work-dir', str(LOCAL_WORK),
    '--drive-dir', str(DRIVE_WORK),
    '--train-shards', str(LOCAL_SHARDS / 'S1_Session_1.npz'), str(LOCAL_SHARDS / 'S1_Session_7.npz'),
    '--val-shards', str(LOCAL_SHARDS / 'S2_Session_1.npz'),
    '--job-id', 'gpu_profile_smoke_w2s',
    '--seed', '0', '--window-s', '2.0', '--max-updates', '100',
    '--checkpoint-every', '50', '--eval-every', '50',
]
subprocess.run(smoke_common + ['--stop-after-updates', '50'], check=True)
subprocess.run(smoke_common + ['--resume'], check=True)

st = json.loads((DRIVE_WORK / 'gpu_profile_smoke_w2s/job_state.json').read_text())
print('Smoke benchmark verified:', json.dumps(st['runtime'], indent=2))
print('Drive SHA-256 verified:', st['drive_verified'], '| Validation:', st['val_metrics'])

In [ ]:
# Cell 5: Stage active scale shards on local disk and run next incomplete campaign job
import subprocess, sys

RUN_CAMPAIGN = False
TARGET_SCALE = 'scale_10pct'  # Options: scale_10pct, scale_30pct, scale_100pct

if RUN_CAMPAIGN:
    subprocess.run([
        sys.executable, '-u', '-m', 'eeg_decoder.phase2_train', 'ingest',
        '--scale', TARGET_SCALE, '--include-dev',
        '--shard-dir', str(LOCAL_SHARDS),
        '--raw-dir', str(LOCAL_RAW),
        '--drive-shard-dir', str(DRIVE_SHARDS),
    ], check=True)
    subprocess.run([
        sys.executable, '-u', '-m', 'eeg_decoder.phase2_train', 'next-job',
        '--scale', TARGET_SCALE,
        '--shard-dir', str(LOCAL_SHARDS),
        '--work-dir', str(LOCAL_WORK),
        '--drive-dir', str(DRIVE_WORK),
        '--execute',
    ], check=True)
else:
    print('Campaign execution paused (RUN_CAMPAIGN=False). Verify Cell 4 smoke benchmark before running full scales.')

In [ ]:
# Cell 6: Standalone Drive progress & learning-curve inspector (runs after runtime reset without GPU/repo)
from pathlib import Path
import json, os
from google.colab import drive

MOUNT = '/content/fri_drive'
if not os.path.ismount(MOUNT):
    drive.mount(MOUNT)
ledger_path = Path(MOUNT) / 'MyDrive/FRI_BCI/phase2_stieger_scaling/campaign_ledger.json'
assert ledger_path.is_file(), f'No ledger yet at {ledger_path}'
jobs = json.loads(ledger_path.read_text()).get('jobs', {})
print(f'Tracked jobs: {len(jobs)}')
for jid, row in sorted(jobs.items()):
    vm = row.get('val_metrics', {})
    rt = row.get('runtime', {})
    print(f"  {jid:28s} | {row.get('status'):11s} | step {row.get('step')}/{row.get('max_updates')} | "
          f"auc_mi_idle={vm.get('val_auc_mi_vs_clean_idle')} | auc_mi_relax={vm.get('val_auc_mi_vs_deliberate_relax')} | "
          f"win/s={rt.get('windows_per_sec')} | drive_ok={row.get('drive_verified')}")